# 기능 3 — 상태 변화 분석 (Google Colab T4)

**처리 범위**
- 기능 1·2 추론 결과 파싱 및 발신자별 집계
- 오늘 vs 직전 7일 비교, 3대 지표 연산
- 종합 상태(🟢🟡🟠⚪) 판정 + 자연어 해석 문구
- 산출물: `sample_report.json`

이 노트북은 기능 1·2 모델 추론부터 기능 3 리포트 생성까지 **전체 경로**를 시연합니다.
기능 3 로직만 단독으로 돌리려면 `state_change_analysis.py` + `sample_chat_log.json`만으로 `python state_change_analysis.py` 실행하면 됩니다 (GPU·모델 불필요).

## CELL 1 — 환경 설정 (T4 런타임)

런타임 → **T4 GPU** 선택 후 실행. 기능 3 연산은 CPU만으로도 동작하지만, 기능 1·2 추론에 GPU를 씁니다.

In [ ]:
# Colab 기본 패키지 (기능 3은 표준 라이브러리만 사용)
import json
import sys
import subprocess
from datetime import date
from pathlib import Path

# 저장소 자동 clone — Run all만으로 기능3 코드(state_change_analysis.py)와
# 입력(sample_chat_log_raw.json)을 확보한다. (이미 있으면 재사용)
REPO_URL = "https://github.com/SARA-MAYO/ChaeOn-AIProgramming.git"
REPO_DIR = Path("/content/ChaeOn-AIProgramming")
if not REPO_DIR.exists():
    print("저장소 clone 중...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("이미 clone된 저장소를 사용합니다.")

# 작업 디렉터리 = 기능3 폴더 (py 파일·입력 json 위치)
WORK_DIR = REPO_DIR / "feature3"
if not WORK_DIR.exists():
    WORK_DIR = Path(".")  # 폴백: 노트북과 같은 폴더에 파일이 있을 때

sys.path.insert(0, str(WORK_DIR))
print("WORK_DIR:", WORK_DIR.resolve())

In [ ]:
import os
from google.colab import drive
import torch, joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── 필수 파일 존재 검사 (폴더뿐 아니라 모델 가중치까지 확인) ──
DRIVE = '/content/drive/MyDrive'
def _has_weights(d):
    return any(os.path.exists(os.path.join(d, w)) for w in ['model.safetensors', 'pytorch_model.bin'])

problems = []
# 모델 폴더: 폴더 존재 + 가중치 파일 존재까지 확인
for name, desc in {
    'chaeon_feature1_checkpoint': '기능1 KcELECTRA 모델  → 기능1 노트북 Run all 필요',
    'chaeon_feature2_model':      '기능2 KoELECTRA 모델  → 기능2 노트북 Run all 필요',
}.items():
    path = os.path.join(DRIVE, name)
    if not os.path.exists(path):
        problems.append(f"  - {name} 폴더 없음            ({desc})")
    elif not _has_weights(path):
        problems.append(f"  - {name} 폴더는 있으나 가중치(model.safetensors) 없음  ({desc})")
# pkl 파일
for name, desc in {
    'svm_model.pkl':  '기능1 SVM 모델          → 기능1 노트북 Run all 필요',
    'vectorizer.pkl': '기능1 TF-IDF 벡터라이저  → 기능1 노트북 Run all 필요',
}.items():
    if not os.path.exists(os.path.join(DRIVE, name)):
        problems.append(f"  - {name} 없음            ({desc})")

if problems:
    raise FileNotFoundError(
        "기능3 실행에 필요한 파일이 Google Drive(MyDrive)에 없거나 불완전합니다:\n"
        + "\n".join(problems)
        + "\n\n→ 부족한 기능의 노트북을 먼저 'Run all' 하면 Drive에 자동 저장됩니다."
    )
print("필수 모델 파일(가중치 포함) 확인 완료")

# 기능1 로드 (Drive 절대경로 사용)
f1_model = AutoModelForSequenceClassification.from_pretrained(
    f'{DRIVE}/chaeon_feature1_checkpoint'
).to(device)
f1_tokenizer = AutoTokenizer.from_pretrained(f'{DRIVE}/chaeon_feature1_checkpoint')
f1_svm = joblib.load(f'{DRIVE}/svm_model.pkl')
f1_vec = joblib.load(f'{DRIVE}/vectorizer.pkl')

# 기능2 로드
f2_model = AutoModelForSequenceClassification.from_pretrained(
    f'{DRIVE}/chaeon_feature2_model'
).to(device)
f2_tokenizer = AutoTokenizer.from_pretrained(f'{DRIVE}/chaeon_feature2_model')

print("기능1·2 모델 로드 완료")

## CELL 2 — 기능 1·2 모델 추론 인터페이스

앞 셀에서 **Drive로부터 자동 로드한** 기능 1·2 모델(`f1_*`, `f2_*`)을 호출해 원문 텍스트를 라벨링합니다.
별도 수정 없이 그대로 실행하면 됩니다.

In [ ]:
import torch
import torch.nn.functional as F

# ──────────────────────────────────────────────────────────────────────────
# [기능 1 | ChaeOn_SVM.ipynb] 변수명
#   model, tokenizer : KcELECTRA (epoch 2 체크포인트)
#   svm_model        : SVC(probability=True)
#   vectorizer       : TfidfVectorizer(max_features=5000)
#   device           : 'cuda' or 'cpu'
#
# [기능 2 | ChaeonAttackModel.ipynb] 변수명
#   model, tokenizer : KoELECTRA-small
#
# ⚠️ 두 모델이 같은 변수명(model, tokenizer)을 사용합니다.
#    기능 1 추론 완료 후 기능 2 모델을 로드하거나,
#    로드 시 변수명을 구분하세요 (예: emotion_model / attack_model).
# ──────────────────────────────────────────────────────────────────────────

def run_feature1_emotion(text: str) -> str:
    # f1_model     : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KcELECTRA 모델 (epoch 2 체크포인트)
    # f1_tokenizer : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KcELECTRA 토크나이저
    # f1_svm       : 새 셀(Cell 1-A)에서 Drive로부터 로드한 SVM 모델 (svm_model.pkl)
    # f1_vec       : 새 셀(Cell 1-A)에서 Drive로부터 로드한 TF-IDF 벡터라이저 (vectorizer.pkl)
    # device       : 새 셀(Cell 1-A)에서 설정한 'cuda' 또는 'cpu'
    f1_model.eval()
    inputs = f1_tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(device)
    with torch.no_grad():
        kc_probs = F.softmax(f1_model(**inputs).logits, dim=-1).cpu().numpy()[0]
    svm_probs = f1_svm.predict_proba(f1_vec.transform([text]))[0]
    prob_pos = (kc_probs[1] * 0.7) + (svm_probs[1] * 0.3)  # KcELECTRA 7 : SVM 3 앙상블
    prob_neg = (kc_probs[0] * 0.7) + (svm_probs[0] * 0.3)
    return "negative" if prob_pos <= prob_neg else "positive"
def run_feature2_aggression(text: str) -> int:
    # f2_model     : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KoELECTRA-small 모델 (checkpoint-5199)
    # f2_tokenizer : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KoELECTRA-small 토크나이저
    f2_model.eval()
    inputs = f2_tokenizer(
        text, return_tensors="pt", padding=True, truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = f2_model(**inputs)
        pred_label = torch.argmax(outputs.logits, dim=-1).item()  # 0=비공격, 1=약한공격, 2=강한공격
    return int(pred_label)
def label_messages(raw_messages: list[dict]) -> list[dict]:
    # run_feature1_emotion, run_feature2_aggression 모두 위 f1_*/f2_* 변수에 의존
    # → 반드시 Cell 1-A(모델 로드 셀) 실행 후에 이 셀을 실행해야 함
    labeled = []
    for msg in raw_messages:
        text = msg["text"]
        labeled.append({
            "message_id":       msg["message_id"],
            "sender_id":        msg["sender_id"],
            "timestamp":        msg["timestamp"],
            "emotion_label":    run_feature1_emotion(text),    # 'negative' 또는 'positive'
            "aggression_label": run_feature2_aggression(text), # 0, 1, 2 정수
        })
    return labeled

## CELL 3 — 입력 데이터 로드

원문 텍스트(`sample_chat_log_raw.json`)가 있으면 기능 1·2 모델로 라벨링하고,
없으면 라벨이 이미 붙은 `sample_chat_log.json`을 그대로 사용합니다.

In [ ]:
import json
from pathlib import Path

# 입력 우선순위:
#   1) sample_chat_log_raw.json — 원문 text만 있는 입력. 기능 1·2 모델로 라벨을 직접 생성(통합 시연).
#   2) sample_chat_log.json     — 이미 라벨이 붙은 입력. 모델 추론 없이 기능 3만 실행.
INPUT_CANDIDATES = ["sample_chat_log_raw.json", "sample_chat_log.json"]
RAW_CHAT_MESSAGES = []
for name in INPUT_CANDIDATES:
    path = WORK_DIR / name
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            RAW_CHAT_MESSAGES = json.load(f)
        print(f"✅ {name} 로드 완료: 총 {len(RAW_CHAT_MESSAGES)}건")
        break
else:
    print("⚠️ 입력 파일이 없습니다. (sample_chat_log_raw.json 또는 sample_chat_log.json 업로드)")

# 기준일은 state_change_analysis.py 내부에서 데이터 최신 날짜를 기준으로 자동 보정되므로
# 여기서는 None으로 설정합니다. (6일 데이터가 무시되지 않게 함)
REFERENCE_DATE = None

## CELL 4 — 기능 3 모듈 로드 (`state_change_analysis.py`)

저장소의 py 파일을 import합니다. **첫 셀에서 저장소가 자동 clone**되므로 별도 업로드·경로 수정이 필요 없습니다.

In [ ]:
from state_change_analysis import (
    MIN_MSG_COUNT,
    MIN_BASELINE_MSG_COUNT,
    run,
    group_messages_by_sender,
    split_time_window,
)

# 입력 데이터에 이미 라벨이 존재하는지 확인 (시연용 JSON 대응)
if RAW_CHAT_MESSAGES and "emotion_label" in RAW_CHAT_MESSAGES[0]:
    print("💡 입력 데이터에 이미 감정/공격성 라벨이 존재합니다. 모델 추론을 생략합니다.")
    labeled_messages = RAW_CHAT_MESSAGES
elif RAW_CHAT_MESSAGES:
    print("💡 텍스트 원문만 존재합니다. 기능 1·2 모델을 구동하여 라벨링을 시작합니다...")
    labeled_messages = label_messages(RAW_CHAT_MESSAGES)
else:
    labeled_messages = []

if labeled_messages:
    print(f"라벨링 완료: {len(labeled_messages)}건")
    print("샘플 1건:", labeled_messages[0])

In [ ]:
# (통합 검증) 기능 1·2 모델이 실제로 라벨링한 결과를 기능 3 입력 파일로 저장.
#   - 여기서 저장되는 sample_chat_log.json = "기능 1·2의 실제 출력 = 기능 3 입력"
#   - 이후 `python state_change_analysis.py` 단독 실행의 입력으로 그대로 쓸 수 있다.
#   - 원문(sample_chat_log_raw.json)으로 모델을 새로 돌린 게 아니라면(=라벨이 이미 있던 입력)
#     덮어쓰지 않는다.
SAVE_REAL_OUTPUT = True  # 모델 실제 출력으로 입력 파일을 갱신하려면 True

made_by_model = bool(RAW_CHAT_MESSAGES) and "emotion_label" not in RAW_CHAT_MESSAGES[0]
if SAVE_REAL_OUTPUT and made_by_model and labeled_messages:
    out_path = WORK_DIR / "sample_chat_log.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(labeled_messages, f, ensure_ascii=False, indent=2)
    print(f"✅ 기능 1·2 실제 출력 저장: {out_path} ({len(labeled_messages)}건)")
    print("   → 이 파일을 다운로드해 저장소의 sample_chat_log.json과 교체하면 '실제 모델 출력'이 됩니다.")
else:
    print("ℹ️ 라벨이 이미 있던 입력이라 저장을 건너뜁니다. (모델 추론 결과만 저장 대상)")

## CELL 5 — 기능 3 실행 → `sample_report.json`

라벨된 메시지로부터 발신자별 최종 리포트(지표 + 상태 + 자연어 해석)를 한 번에 생성합니다.

In [ ]:
import json

# 발신자별 분석 + 리포트 생성 (지표 계산 → 상태 판정 → 자연어 해석까지 한 번에)
reports = run(labeled_messages, reference_date=REFERENCE_DATE)

OUTPUT_PATH = WORK_DIR / "sample_report.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(reports, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {OUTPUT_PATH}")
print(json.dumps(reports, ensure_ascii=False, indent=2))

In [ ]:
# sample_report.json → Google Drive 백업
import shutil

shutil.copy(
    str(WORK_DIR / "sample_report.json"),
    "/content/drive/MyDrive/sample_report.json"
)
print("sample_report.json Drive 저장 완료")

## CELL 6 — 교차 검증 (자체 점검)

`split_time_window` 분리 건수와 `data_sufficient` 가드레일 동작을 확인합니다.

In [ ]:
from state_change_analysis import group_messages_by_sender

grouped = group_messages_by_sender(labeled_messages)

for sender_id, msgs in grouped.items():
    today, baseline = split_time_window(msgs, REFERENCE_DATE)
    result = next(r for r in reports if r["sender_id"] == sender_id)
    print(f"[{sender_id}] today={len(today)}, baseline={len(baseline)}, "
          f"data_sufficient={result['data_sufficient']}, "
          f"overall={result['overall']['state']}")

# 가드레일 상수 확인
assert MIN_MSG_COUNT == 10
assert MIN_BASELINE_MSG_COUNT == 10
print("\n가드레일 상수 OK")